# SwinV2 — Swin Transformer V2

**Paper:** Z. Liu, H. Hu, Y. Lin, Z. Yao, Z. Wei, S. Ning, Y. Cao, Z. Zhang, L. Dong, F. Wei, and B. Guo,
"Swin Transformer V2: Scaling Up Capacity and Resolution," in *Proc. IEEE/CVF CVPR*, 2022, pp. 12009–12019.
arXiv: [2111.09883](https://arxiv.org/abs/2111.09883)

> **One-line summary:** A hierarchical window-attention transformer with three
> scaling-friendly upgrades — residual post-norm, log-CPB position bias, and
> scaled cosine attention — enabling models up to 3 billion parameters trained
> on images up to 1536×1536.


## Architecture Pipeline (ASCII)

```
Input Image [B, 3, H, W]
        │
        ▼
┌─────────────────────┐
│   Patch Embedding   │  4×4 stride Conv → Linear projection
│   [B, H/4, W/4, C] │
└──────────┬──────────┘
           │
     ┌─────▼──────┐
     │   Stage 0  │  depth=2,  output [B, H/4,  W/4,  C]
     │  W-MSA + SW-MSA blocks  │
     └─────┬──────┘
           │  Patch Merge (2C)
     ┌─────▼──────┐
     │   Stage 1  │  depth=2,  output [B, H/8,  W/8,  2C]
     └─────┬──────┘
           │  Patch Merge (4C)
     ┌─────▼──────┐
     │   Stage 2  │  depth=6–42, output [B, H/16, W/16, 4C]
     └─────┬──────┘
           │  Patch Merge (8C)
     ┌─────▼──────┐
     │   Stage 3  │  depth=2,  output [B, H/32, W/32, 8C]
     └─────┬──────┘
           │
    Global Avg Pool
           │
    pooled_output [B, 8C]

─────────────────────────────────────────
 Single SwinV2 Block Detail
─────────────────────────────────────────

  x [B, H, W, C]
       │
  [Cyclic Shift by win//2] ← only shifted blocks
       │
  Window Partition → [B*nW, win², C]
       │
  Scaled Cosine Attention
    Q, K, V = Linear(x)
    A = cos(Q,K) / τ  +  LogCPB(Δrow, Δcol)
    out = softmax(A) · V
       │
  Window Reverse
       │
  [Reverse Cyclic Shift] ← only shifted blocks
       │
  Residual: x = LN(x + attn_out)   ← Post-Norm
       │
  FFN: Linear(4C)→GELU→Linear(C)
       │
  Residual: x = LN(x + ffn_out)    ← Post-Norm
```


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import numpy as np

fig = plt.figure(figsize=(18, 14))

# ── Left panel: model-level hierarchy ──────────────────────────────────────
ax1 = fig.add_axes([0.02, 0.05, 0.32, 0.90])
ax1.set_xlim(0, 4)
ax1.set_ylim(0, 11)
ax1.axis('off')
ax1.set_title('SwinV2 Hierarchical Architecture', fontsize=12, fontweight='bold', pad=8)

stages = [
    ('Input\n[B, 3, 224, 224]',    10.2, '#e8eaf6'),
    ('Patch Embed\n[B, 56×56, C]', 9.0,  '#c5cae9'),
    ('Stage 0  depth=2\n[B, 56×56, C]',  7.6,  '#90caf9'),
    ('Patch Merge → 2C',            6.6,  '#b0bec5'),
    ('Stage 1  depth=2\n[B, 28×28, 2C]', 5.6,  '#80deea'),
    ('Patch Merge → 4C',            4.6,  '#b0bec5'),
    ('Stage 2  depth=6\n[B, 14×14, 4C]', 3.6,  '#a5d6a7'),
    ('Patch Merge → 8C',            2.6,  '#b0bec5'),
    ('Stage 3  depth=2\n[B,  7× 7, 8C]', 1.6,  '#ffe082'),
    ('Global Avg Pool\npooled [B, 8C]',  0.4,  '#ffccbc'),
]

for label, y, color in stages:
    box = FancyBboxPatch((0.3, y - 0.45), 3.4, 0.85,
                         boxstyle='round,pad=0.05', facecolor=color, edgecolor='#455a64', linewidth=1.2)
    ax1.add_patch(box)
    ax1.text(2.0, y, label, ha='center', va='center', fontsize=8.5)

for i in range(len(stages) - 1):
    y1 = stages[i][1] - 0.5
    y2 = stages[i+1][1] + 0.5
    ax1.annotate('', xy=(2.0, y2), xytext=(2.0, y1),
                 arrowprops=dict(arrowstyle='->', color='#455a64', lw=1.5))

# ── Middle panel: block internals ──────────────────────────────────────────
ax2 = fig.add_axes([0.36, 0.05, 0.30, 0.90])
ax2.set_xlim(0, 4)
ax2.set_ylim(0, 11)
ax2.axis('off')
ax2.set_title('SwinV2 Block Internals (Post-Norm)', fontsize=12, fontweight='bold', pad=8)

block_steps = [
    ('Input x [B, H, W, C]', 10.3, '#e8eaf6'),
    ('Cyclic Shift\n(shifted blocks only)', 9.0, '#fff9c4'),
    ('Window Partition\n→ [B·nW, win², C]', 7.8, '#c5cae9'),
    ('Scaled Cosine Attn\ncos(Q,K)/τ + LogCPB', 6.4, '#bbdefb'),
    ('Window Reverse\n→ [B, H, W, C]', 5.2, '#c5cae9'),
    ('Reverse Cyclic Shift\n(shifted blocks only)', 4.0, '#fff9c4'),
    ('Post-Norm Residual\nLN(x + attn_out)', 2.8, '#dcedc8'),
    ('MLP: 4× expand, GELU', 1.8, '#ffe0b2'),
    ('Post-Norm Residual\nLN(x + mlp_out)', 0.6, '#dcedc8'),
]

for label, y, color in block_steps:
    box = FancyBboxPatch((0.2, y - 0.48), 3.6, 0.88,
                         boxstyle='round,pad=0.05', facecolor=color, edgecolor='#455a64', linewidth=1.1)
    ax2.add_patch(box)
    ax2.text(2.0, y, label, ha='center', va='center', fontsize=8.5)

for i in range(len(block_steps) - 1):
    y1 = block_steps[i][1] - 0.5
    y2 = block_steps[i+1][1] + 0.5
    ax2.annotate('', xy=(2.0, y2), xytext=(2.0, y1),
                 arrowprops=dict(arrowstyle='->', color='#455a64', lw=1.4))

# ── Right panel: window shifting visualisation ──────────────────────────────
ax3 = fig.add_axes([0.68, 0.35, 0.30, 0.58])
ax3.set_xlim(-0.5, 8.5)
ax3.set_ylim(-0.5, 8.5)
ax3.set_aspect('equal')
ax3.set_title('Shifted Window (8×8 map, win=4)', fontsize=10, fontweight='bold')
ax3.axis('off')

grid_colors_orig = [
    '#bbdefb', '#bbdefb', '#bbdefb', '#bbdefb', '#c8e6c9', '#c8e6c9', '#c8e6c9', '#c8e6c9',
    '#bbdefb', '#bbdefb', '#bbdefb', '#bbdefb', '#c8e6c9', '#c8e6c9', '#c8e6c9', '#c8e6c9',
    '#bbdefb', '#bbdefb', '#bbdefb', '#bbdefb', '#c8e6c9', '#c8e6c9', '#c8e6c9', '#c8e6c9',
    '#bbdefb', '#bbdefb', '#bbdefb', '#bbdefb', '#c8e6c9', '#c8e6c9', '#c8e6c9', '#c8e6c9',
    '#ffe082', '#ffe082', '#ffe082', '#ffe082', '#ffccbc', '#ffccbc', '#ffccbc', '#ffccbc',
    '#ffe082', '#ffe082', '#ffe082', '#ffe082', '#ffccbc', '#ffccbc', '#ffccbc', '#ffccbc',
    '#ffe082', '#ffe082', '#ffe082', '#ffe082', '#ffccbc', '#ffccbc', '#ffccbc', '#ffccbc',
    '#ffe082', '#ffe082', '#ffe082', '#ffe082', '#ffccbc', '#ffccbc', '#ffccbc', '#ffccbc',
]
for row in range(8):
    for col in range(8):
        idx = row * 8 + col
        rect = mpatches.FancyBboxPatch((col + 0.05, row + 0.05), 0.9, 0.9,
                                       boxstyle='square,pad=0.0',
                                       facecolor=grid_colors_orig[idx],
                                       edgecolor='#78909c', linewidth=0.6)
        ax3.add_patch(rect)

for i in range(0, 9, 4):
    ax3.plot([i, i], [0, 8], color='#1565c0', lw=2.0)
    ax3.plot([0, 8], [i, i], color='#1565c0', lw=2.0)

ax3.text(4.0, -0.35, 'Standard windows (4 non-overlapping windows)', ha='center', fontsize=8)

# Shifted window panel
ax4 = fig.add_axes([0.68, 0.05, 0.30, 0.28])
ax4.set_xlim(-0.5, 8.5)
ax4.set_ylim(-0.5, 8.5)
ax4.set_aspect('equal')
ax4.set_title('After Cyclic Shift (shift=2)', fontsize=10, fontweight='bold')
ax4.axis('off')

shift_colors = [
    '#ef9a9a', '#ef9a9a', '#ce93d8', '#ce93d8', '#ef9a9a', '#ef9a9a', '#ce93d8', '#ce93d8',
    '#ef9a9a', '#ef9a9a', '#ce93d8', '#ce93d8', '#ef9a9a', '#ef9a9a', '#ce93d8', '#ce93d8',
    '#80cbc4', '#80cbc4', '#fff176', '#fff176', '#80cbc4', '#80cbc4', '#fff176', '#fff176',
    '#80cbc4', '#80cbc4', '#fff176', '#fff176', '#80cbc4', '#80cbc4', '#fff176', '#fff176',
    '#ef9a9a', '#ef9a9a', '#ce93d8', '#ce93d8', '#ef9a9a', '#ef9a9a', '#ce93d8', '#ce93d8',
    '#ef9a9a', '#ef9a9a', '#ce93d8', '#ce93d8', '#ef9a9a', '#ef9a9a', '#ce93d8', '#ce93d8',
    '#80cbc4', '#80cbc4', '#fff176', '#fff176', '#80cbc4', '#80cbc4', '#fff176', '#fff176',
    '#80cbc4', '#80cbc4', '#fff176', '#fff176', '#80cbc4', '#80cbc4', '#fff176', '#fff176',
]
for row in range(8):
    for col in range(8):
        idx = row * 8 + col
        rect = mpatches.FancyBboxPatch((col + 0.05, row + 0.05), 0.9, 0.9,
                                       boxstyle='square,pad=0.0',
                                       facecolor=shift_colors[idx],
                                       edgecolor='#78909c', linewidth=0.6)
        ax4.add_patch(rect)

for i in range(0, 9, 4):
    ax4.plot([i, i], [0, 8], color='#b71c1c', lw=2.0)
    ax4.plot([0, 8], [i, i], color='#b71c1c', lw=2.0)

ax4.text(4.0, -0.35, '4 shifted windows cross original boundaries (masked)', ha='center', fontsize=7.5)

plt.suptitle('Swin Transformer V2 — Architecture Diagrams', fontsize=14, fontweight='bold', y=0.995)
plt.savefig('swinv2_architecture.png', dpi=140, bbox_inches='tight')
plt.show()
print('Architecture diagram saved.')


## Key Innovations (SwinV1 → SwinV2)

### 1. Residual Post-Norm

**SwinV1 (pre-norm):**
$$x' = x + \text{Attn}(\text{LN}(x))$$
$$x'' = x' + \text{FFN}(\text{LN}(x'))$$

**SwinV2 (post-norm):**
$$x' = \text{LN}\bigl(x + \text{Attn}(x)\bigr)$$
$$x'' = \text{LN}\bigl(x' + \text{FFN}(x')\bigr)$$

*Why it matters:* In very deep networks (e.g. 48-layer SwinV2-G), pre-norm causes
activation magnitudes to grow with depth, making training unstable.
Post-norm normalises after each residual, keeping magnitudes controlled throughout.

---

### 2. Log-Continuous Position Bias (Log-CPB)

**SwinV1** learned a fixed position-bias *table* of shape
$[(2 \cdot \text{win} - 1)^2,\ \text{heads}]$.
This table is tied to the training window size — changing the window at
inference requires discarding or interpolating learned biases.

**SwinV2** replaces the table with a tiny MLP:

$$\hat{\Delta} = \text{sign}(\Delta)\cdot\log_{}(1 + |\Delta|)$$
$$B(\Delta_r, \Delta_c) = 16\cdot\sigma\!\left(\text{MLP}(\hat{\Delta}_r,\, \hat{\Delta}_c)\right)$$

The MLP is a 2-layer network $\mathbb{R}^2 \to \mathbb{R}^{512} \to \mathbb{R}^{\text{heads}}$.
Because the MLP takes *continuous* log-space coordinates rather than discrete integers,
it generalises smoothly to unseen window sizes at inference.

---

### 3. Scaled Cosine Attention

**Standard dot-product:**
$$A_{ij} = \frac{\mathbf{q}_i^\top \mathbf{k}_j}{\sqrt{d}} + B_{ij}$$

**SwinV2 scaled cosine:**
$$A_{ij} = \frac{\cos(\mathbf{q}_i, \mathbf{k}_j)}{\tau} + B_{ij}$$

where $\tau = \exp(\log \hat{\tau})$ is a *learnable per-head temperature*,
initialised to $\log(10)$.

*Why cosine?*  In large models, dot-product magnitudes grow with embedding dimension,
leading to peaked (low-entropy) attention distributions — a small number of tokens
dominate. Cosine similarity is normalised to $[-1, 1]$, and $\tau$ provides a
tunable sharpness independent of dimension scale.

---

### 4. Shifted Window Attention Mask

After cyclic shift by $(\text{win}//2,\ \text{win}//2)$, each window may contain
tokens from **up to 4 different** original windows. An attention mask
$M \in \{0, -\infty\}^{\text{win}^2 \times \text{win}^2}$ is added
before softmax so tokens from different original regions cannot attend to each other:

$$M_{ij} = \begin{cases} 0 & \text{if region}(i) = \text{region}(j) \\ -100 & \text{otherwise} \end{cases}$$

This preserves the intended local receptive field while enabling cross-window
connections in the NEXT block (which will not be shifted).


## Code Walkthrough

### Log-CPB MLP (registry path: `hale_core.nn.vision.swin.swinv2.LogContinuousPositionBias`)

```python
class LogContinuousPositionBias(nn.Module):
    def __init__(self, num_heads):
        self.cpb_mlp = nn.Sequential(
            nn.Linear(2, 512, bias=True),
            nn.ReLU(inplace=True),
            nn.Linear(512, num_heads, bias=False),
        )

    def _compute_bias(self, window_size):
        coords_h = torch.arange(window_size)
        coords_w = torch.arange(window_size)
        coords = torch.stack(torch.meshgrid(coords_h, coords_w, indexing='ij'))
        coords_flat = coords.flatten(1)              # [2, win*win]
        rel = coords_flat[:, :, None] - coords_flat[:, None, :]
        rel = rel.permute(1, 2, 0).float()           # [win², win², 2]

        # Log-compress: sign(d)·log(1+|d|) keeps values in ≈[-3.2, 3.2]
        rel_log = rel.sign() * torch.log1p(rel.abs())
        rel_log = rel_log / math.log(window_size) * 8.0   # normalise

        bias = 16.0 * self.cpb_mlp(rel_log).sigmoid()     # [win², win², heads]
        return bias.permute(2, 0, 1)                       # [heads, win², win²]
```

### Scaled Cosine Attention (`SwinV2WindowAttention.forward`)

```python
def forward(self, x, mask=None):
    B, N, C = x.shape
    qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
    q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)   # [B, heads, N, d]

    # Normalise to unit sphere, then scale by learnable temperature
    q = F.normalize(q, dim=-1)
    k = F.normalize(k, dim=-1)
    tau = self.logit_scale.exp().clamp(max=math.log(100))
    attn = (q @ k.transpose(-2, -1)) * tau             # cosine / tau

    # Add Log-CPB relative position bias
    bias = self.cpb(self.window_size)                  # [heads, win², win²]
    attn = attn + bias.unsqueeze(0)

    # Shifted-window mask: adds -100 to blocked token pairs
    if mask is not None:
        attn = attn + mask.unsqueeze(1)
    ...
```

### Patch Merging

```python
class PatchMerging(nn.Module):
    # Doubles channels while halving resolution (like a strided pooling)
    def forward(self, x):                 # x: [B, H, W, C]
        x0 = x[:, 0::2, 0::2, :]         # top-left
        x1 = x[:, 1::2, 0::2, :]         # bottom-left
        x2 = x[:, 0::2, 1::2, :]         # top-right
        x3 = x[:, 1::2, 1::2, :]         # bottom-right
        x = torch.cat([x0, x1, x2, x3], dim=-1)   # [B, H/2, W/2, 4C]
        return self.proj(self.norm(x))             # → [B, H/2, W/2, 2C]
```

### Model Variants

| Name | Window | Channels C | Depths | Params |
|------|--------|-----------|--------|--------|
| swinv2_tiny | 8 | 96 | [2,2,6,2] | ~28 M |
| swinv2_small | 8 | 96 | [2,2,18,2] | ~50 M |
| swinv2_base | 8 | 128 | [2,2,18,2] | ~88 M |
| swinv2_large | 16 | 192 | [2,2,18,2] | ~197 M |
| swinv2_giant | 16 | 192 | [2,2,42,2] | ~3 B |


In [ ]:
import sys
sys.path.insert(0, '../../src')
import torch
from hale_core.nn.vision import build_vision_backbone

# ── Build SwinV2-Tiny and trace shapes through stages ──────────────────────
backbone = build_vision_backbone('swinv2_tiny', img_size=224)
backbone.eval()

print(f'Model: swinv2_tiny')
print(f'Parameters: {backbone.num_parameters():,}')
print(f'Embed dim:  {backbone.embed_dim}')
print()

# Register hooks to print intermediate shapes
shapes = {}
def make_hook(name):
    def hook(module, inp, out):
        if isinstance(out, torch.Tensor):
            shapes[name] = tuple(out.shape)
    return hook

backbone.patch_embed.register_forward_hook(make_hook('patch_embed'))
for i, stage in enumerate(backbone.stages):
    stage.register_forward_hook(make_hook(f'stage_{i}'))

x = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    out = backbone(x)

print('Shape trace (channels-last, HaleBlocks uses channels-first in feature_maps):')
for name, shape in shapes.items():
    print(f'  {name:15s} → {shape}')
print()
print('BackboneOutput:')
print(f'  pooled_output : {tuple(out.pooled_output.shape)}')
print(f'  feature_maps  : {[tuple(fm.shape) for fm in out.feature_maps]}')
print(f'  patch_tokens  : {out.patch_tokens}')

# Test freeze API
backbone.freeze_up_to_stage(1)
trainable = backbone.num_parameters(trainable_only=True)
total    = backbone.num_parameters()
print(f'\nAfter freeze_up_to_stage(1):')
print(f'  Trainable: {trainable:,} / {total:,}  ({100*trainable/total:.1f}%)')


## Paper Citation & Related Work

### Primary Reference

> Z. Liu, H. Hu, Y. Lin, Z. Yao, Z. Wei, S. Ning, Y. Cao, Z. Zhang, L. Dong, F. Wei, and B. Guo,
> "Swin Transformer V2: Scaling Up Capacity and Resolution,"
> in *Proceedings of the IEEE/CVF Conference on Computer Vision and Pattern Recognition (CVPR)*,
> 2022, pp. 12009–12019.
> **arXiv:** https://arxiv.org/abs/2111.09883

### SwinV1 (predecessor)

> Z. Liu, Y. Lin, Y. Cao, H. Hu, Y. Wei, Z. Zhang, S. Lin, and B. Guo,
> "Swin Transformer: Hierarchical Vision Transformer using Shifted Windows,"
> in *ICCV*, 2021. arXiv: 2103.14030

### Related Architectures

| Model | Key Idea | Reference |
|-------|----------|----------|
| ViT | Global patch self-attention | Dosovitskiy et al., ICLR 2021 |
| Twins | Alternative to shifted windows using PVT-style pooling | Chu et al., NeurIPS 2021 |
| CSWin | Cross-shaped window attention | Dong et al., CVPR 2022 |
| FocalTransformer | Focal attention (local fine + global coarse) | Yang et al., NeurIPS 2021 |
| MaxViT | Block + grid attention (this repo) | Tu et al., ECCV 2022 |

### Key Ablations from Paper

- Residual post-norm: +0.1 ImageNet top-1, critical for 3B-param stability
- Log-CPB vs. bicubic interpolation of SwinV1 table: +0.3 top-1 at different resolutions
- Scaled cosine vs. dot-product: +0.5 top-1 for SwinV2-G
- SwinV2-G achieves **90.17%** top-1 on ImageNet-1K (224px), **63.1** box AP on COCO
